# Practice SQL Database types

Fill out the code, as necessary, below based on the directions in Canvas.

As an independent data science consultant tasked with designing and implementing a database for a school, your approach must balance practicality with strategic foresight. This task is not just about creating a database that functions correctly today, but about developing a system that will remain effective as the school grows and its needs evolve. The primary objective is to build a robust and flexible database that can seamlessly manage essential information, including student and staff contact details, academic records, class rosters, and attendance logs.

To achieve this, you'll need to carefully consider the structure and relationships of the data from the outset. The database should be normalized to avoid redundancy and ensure data integrity, which will make it easier to maintain, query, and scale in the future. The design process involves identifying the key entities (such as students, staff, courses, and grades) and mapping out how these entities will interact with one another within the database. This requires not only a strong understanding of the data itself but also foresight into how the data might need to be accessed, updated, and analyzed over time.

In this particular implementation, you are working within a Python environment, leveraging the `sqlite3` module to create and manipulate the database. SQLite is a powerful, lightweight, and self-contained SQL database engine that is well-suited for this task due to its simplicity and ease of integration with Python. It allows you to create a local database file (`school.sqlite`) that can be easily managed and queried directly from your Python scripts.

The process begins with setting up the database file and establishing a connection to it. This is followed by designing the schema, which involves creating tables to represent the different entities within the school’s ecosystem. Each table must be carefully structured with appropriate columns and data types to capture the necessary information. For instance, the `contactInfo` table should include columns for first names, last names, roles (indicating whether the person is a student or staff member), and address details. Each entry in the table is uniquely identified by a primary key, ensuring that records can be accurately referenced and related across different tables.

As you proceed with populating the tables, it's crucial to write efficient and secure SQL queries to insert, update, and retrieve data. This process often involves iterating over datasets (like lists of dictionaries) and carefully formatting SQL commands to prevent errors such as SQL injection or data corruption. Additionally, you must implement measures to handle potential issues, such as duplicate records or the need for data updates, ensuring the database remains accurate and reliable.

Beyond the immediate implementation, it's essential to keep in mind the future needs of the school. This includes designing the database to accommodate new features, such as additional tables for attendance tracking or extracurricular activities, and ensuring that it can handle increasing data volume as the school grows. Moreover, the database should be structured in a way that allows for easy integration with other systems, such as learning management systems or student information systems, which may be added in the future.

In summary, your role as a data science consultant involves not only the technical creation of the database but also strategic planning to ensure the system is scalable, secure, and able to meet the long-term needs of the school. By carefully designing the database schema, implementing best practices in SQL, and considering future expansions, you will create a robust foundation that supports the school’s operational and analytical requirements.

In [2]:
# Import necessary packages
import sqlite3
import pandas as pd

# Create the database school.sqlite
conn = sqlite3.Connection('school.sqlite')

In [3]:
cur = conn.cursor()
cur.execute("""CREATE TABLE contactInfo (
                                        userId INTEGER PRIMARY KEY,
                                        firstName TEXT,
                                        lastName TEXT,
                                        role TEXT,
                                        telephone INTEGER,
                                        street TEXT,
                                        city TEXT,
                                        state TEXT,
                                        zipcode TEXT
                                        );
            """)

In [4]:
# Load the list of dictionaries; just run this cell
import pickle

with open('contact_list.pickle', 'rb') as f:
    contacts = pickle.load(f)

# Iterate over the contact list and populate the contactInfo table here
for contact in contacts:
    firstName = contact['firstName']
    lastName = contact['lastName']
    role = contact['role']
    telephone  = contact['telephone ']
    street = contact['street']
    city = contact['city']
    state = contact['state']
    zipcode  = contact['zipcode ']
    cur.execute("""INSERT INTO contactInfo (firstName, lastName, role, telephone, street, city, state, zipcode) 
                  VALUES ('{}', '{}', '{}', '{}', '{}', '{}', '{}', '{}');
                """.format(firstName, lastName, role, telephone, street, city, state, zipcode) )

In [5]:
cur.execute("""SELECT * 
               FROM contactInfo;""")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]
df

,userId,firstName,lastName,role,telephone,street,city,state,zipcode
0,1,Christine,Holden,staff,2035687697,1672 Whitman Court,Stamford,CT,06995
1,2,Christopher,Warren,student,2175150957,1935 University Hill Road,Champaign,IL,61938
2,3,Linda,Jacobson,staff,4049446441,479 Musgrave Street,Atlanta,GA,30303
3,4,Andrew,Stepp,student,7866419252,2981 Lamberts Branch Road,Hialeah,Fl,33012
4,5,Jane,Evans,student,3259909290,1461 Briarhill Lane,Abilene,TX,79602
5,6,Jane,Evans,student,3259909290,1461 Briarhill Lane,Abilene,TX,79602
6,7,Mary,Raines,student,9075772295,3975 Jerry Toth Drive,Ninilchik,AK,99639
7,8,Ed,Lyman,student,5179695576,3478 Be Sreet,Lansing,MI,48933


In [6]:
conn.commit()

In [7]:
# Create the grades table
cur.execute("""CREATE TABLE grades (
                                    userId INTEGER NOT NULL,
                                    courseId INTEGER NOT NULL,
                                    grade INTEGER,
                                    PRIMARY KEY(userId, courseId)
                                    );
            """)

In [8]:
# Find the duplicate entry
cur.execute("""SELECT firstName, lastName, telephone, COUNT(*) 
               FROM contactInfo
               GROUP BY firstName, lastName, telephone
               HAVING COUNT(*) > 1;""").fetchall()

[('Jane', 'Evans', 3259909290, 2)]

In [9]:
# Delete the duplicate entry
cur.execute("""DELETE FROM contactInfo 
               WHERE telephone = 3259909290;""")

# Check that the duplicate entry was removed 
cur.execute("""SELECT firstName, lastName, telephone, COUNT(*) 
               FROM contactInfo
               GROUP BY firstName, lastName, telephone
               HAVING COUNT(*) > 1;""").fetchall()

[]

In [10]:
# Update Ed's address
cur.execute("""UPDATE contactInfo
               SET street = "2910 Simpson Avenue",
                   city = 'York',
                   state = 'PA',
                   zipcode = '17403'
               WHERE firstName = "Ed" AND lastName = "Lyman";""")

# Query the database to ensure the change was made
cur.execute("""SELECT * 
               FROM contactInfo;""")
df = pd.DataFrame(cur.fetchall())
df.columns = [x[0] for x in cur.description]
df

,userId,firstName,lastName,role,telephone,street,city,state,zipcode
0,1,Christine,Holden,staff,2035687697,1672 Whitman Court,Stamford,CT,06995
1,2,Christopher,Warren,student,2175150957,1935 University Hill Road,Champaign,IL,61938
2,3,Linda,Jacobson,staff,4049446441,479 Musgrave Street,Atlanta,GA,30303
3,4,Andrew,Stepp,student,7866419252,2981 Lamberts Branch Road,Hialeah,Fl,33012
4,7,Mary,Raines,student,9075772295,3975 Jerry Toth Drive,Ninilchik,AK,99639
5,8,Ed,Lyman,student,5179695576,2910 Simpson Avenue,York,PA,17403


In [11]:
conn.commit()

In [12]:
# Remember to close the connection when you are done
conn.close()